# Foldcomp to FoldTree2 FASTA Benchmark

This notebook benchmarks the Foldcomp conversion path used by `mk1_Encoder.encode_foldcomp_fasta` and estimates total runtime for converting an entire Foldcomp database (for example under `/mnt/data1/foldcomp`).

It runs a sampled conversion multiple times, computes throughput, and extrapolates to the full database using the lookup file entry count.

In [1]:
from pathlib import Path
import math
import random
import statistics
import time

import pandas as pd
import torch

In [2]:
#use autoreload to reload modules when they are edited
from IPython import get_ipython
get_ipython().run_line_magic('load_ext', 'autoreload')
get_ipython().run_line_magic('autoreload', '2')

In [3]:
def resolve_foldcomp_db(db_input: str) -> str:
    """Resolve Foldcomp DB basename from a basename path, .lookup file, or directory."""
    p = Path(db_input)

    if p.is_dir():
        lookups = sorted(p.glob('*.lookup'))
        if len(lookups) == 0:
            raise FileNotFoundError(f'No .lookup files found in directory: {p}')
        if len(lookups) > 1:
            raise ValueError(
                f'Multiple .lookup files found in {p}. Please pass a specific DB basename. '
                f'Found: {[x.name for x in lookups]}'
            )
        return str(lookups[0].with_suffix(''))

    if p.suffix == '.lookup' and p.exists():
        return str(p.with_suffix(''))

    if p.exists() and Path(str(p) + '.lookup').exists():
        return str(p)

    if (not p.exists()) and Path(str(p) + '.lookup').exists():
        return str(p)

    raise FileNotFoundError(
        f'Could not resolve Foldcomp DB basename from {db_input}. '
        f'Expected a basename with companion .lookup file or a directory containing one .lookup.'
    )


def count_and_sample_lookup_ids(lookup_path: str, sample_size: int, seed: int = 42):
    """Count entries and select a uniform sample using reservoir sampling without loading full IDs into memory."""
    rng = random.Random(seed)
    sample = []
    total = 0

    with open(lookup_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split()
            if len(parts) < 2:
                continue
            entry_id = parts[1]
            total += 1

            if len(sample) < sample_size:
                sample.append(entry_id)
            else:
                j = rng.randint(1, total)
                if j <= sample_size:
                    sample[j - 1] = entry_id

    if total == 0:
        raise ValueError(f'No valid entries found in lookup file: {lookup_path}')

    if len(sample) == 0:
        raise ValueError('Sample is empty. Increase sample_size or validate the lookup file.')

    return total, sample


def format_duration(seconds: float) -> str:
    seconds = int(round(seconds))
    days, rem = divmod(seconds, 86400)
    hours, rem = divmod(rem, 3600)
    minutes, sec = divmod(rem, 60)
    if days > 0:
        return f'{days}d {hours}h {minutes}m {sec}s'
    if hours > 0:
        return f'{hours}h {minutes}m {sec}s'
    if minutes > 0:
        return f'{minutes}m {sec}s'
    return f'{sec}s'

In [4]:
def load_encoder(model_path: str, device: None):
    model_path = str(model_path)
    if not Path(model_path).exists():
        raise FileNotFoundError(f'Model not found: {model_path}')

    if device is None:
        torch_device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    else:
        torch_device = torch.device(device)

    encoder = torch.load(model_path, map_location=torch_device, weights_only=False)
    encoder = encoder.to(torch_device)
    encoder.device = torch_device
    encoder.eval()

    return encoder, torch_device


def benchmark_foldcomp_sample(
    encoder,
    foldcomp_db: str,
    sample_ids: list[str],
    out_dir: str,
    repeats: int = 3,
    chunk_size: int = 1024,
    queue_size: int = 4,
    batch_size: int = 16,
    cache_size: int = 0,
    verbose: bool = False,
) -> pd.DataFrame:
    out_path = Path(out_dir)
    out_path.mkdir(parents=True, exist_ok=True)

    records = []
    for run_idx in range(1, repeats + 1):
        fasta_path = out_path / f'sample_run_{run_idx}.fasta'

        start = time.perf_counter()
        encoder.encode_foldcomp_fasta(
            foldcomp_db=foldcomp_db,
            filename=str(fasta_path),
            ids=sample_ids,
            max_structures=None,
            chunk_size=chunk_size,
            queue_size=queue_size,
            batch_size=batch_size,
            cache_size=cache_size,
            replace=True,
            alphabet=None,
            verbose=verbose,
        )
        elapsed = time.perf_counter() - start

        converted = len(sample_ids)
        throughput = converted / elapsed if elapsed > 0 else float('inf')
        fasta_size_mb = fasta_path.stat().st_size / (1024 ** 2)

        records.append({
            'run': run_idx,
            'structures': converted,
            'seconds': elapsed,
            'structures_per_second': throughput,
            'fasta_size_mb': fasta_size_mb,
            'fasta_path': str(fasta_path),
        })

    return pd.DataFrame.from_records(records)


def estimate_total_runtime(total_entries: int, throughputs: list[float]):
    mean_tp = statistics.mean(throughputs)

    if len(throughputs) > 1:
        std_tp = statistics.stdev(throughputs)
    else:
        std_tp = 0.0

    margin = 1.96 * (std_tp / math.sqrt(max(1, len(throughputs))))
    tp_low = max(1e-9, mean_tp - margin)
    tp_high = mean_tp + margin

    mean_seconds = total_entries / mean_tp
    worst_seconds = total_entries / tp_low
    best_seconds = total_entries / tp_high

    return {
        'mean_throughput': mean_tp,
        'std_throughput': std_tp,
        'throughput_ci95_low': tp_low,
        'throughput_ci95_high': tp_high,
        'eta_mean_seconds': mean_seconds,
        'eta_worst_seconds': worst_seconds,
        'eta_best_seconds': best_seconds,
    }

In [5]:
# ---- User configuration ----
FOLDCOMP_INPUT = '/mnt/data1/foldcomp'
MODEL_PATH = '/home/dmoi/projects/foldtree2/models/production/30char_minimal_decoder/final_30char_contacts_aa_encoder_full_epoch_52.pt'
DEVICE = None  # e.g. 'cuda:0' or 'cpu'

SAMPLE_SIZE = 1000
REPEATS = 3
SEED = 42

CHUNK_SIZE = 10
QUEUE_SIZE = 10
BATCH_SIZE = 10
CACHE_SIZE = 0

OUT_DIR = 'tmp/foldcomp_benchmark'

print('Configured.')
print(f'Foldcomp input: {FOLDCOMP_INPUT}')
print(f'Model path: {MODEL_PATH}')
print(f'Sample size x repeats: {SAMPLE_SIZE} x {REPEATS}')

Configured.
Foldcomp input: /mnt/data1/foldcomp
Model path: /home/dmoi/projects/foldtree2/models/production/30char_minimal_decoder/final_30char_contacts_aa_encoder_full_epoch_52.pt
Sample size x repeats: 1000 x 3


In [6]:
foldcomp_db = resolve_foldcomp_db(FOLDCOMP_INPUT)
lookup_path = f'{foldcomp_db}.lookup'

total_entries, sample_ids = count_and_sample_lookup_ids(
    lookup_path=lookup_path,
    sample_size=SAMPLE_SIZE,
    seed=SEED,
)

print(f'Resolved Foldcomp DB basename: {foldcomp_db}')
print(f'Lookup file: {lookup_path}')
print(f'Total entries in DB: {total_entries:,}')
print(f'Sampled entries for timing: {len(sample_ids):,}')

Resolved Foldcomp DB basename: /mnt/data1/foldcomp/afdb_swissprot_v4
Lookup file: /mnt/data1/foldcomp/afdb_swissprot_v4.lookup
Total entries in DB: 542,378
Sampled entries for timing: 1,000


In [11]:
encoder, active_device = load_encoder(MODEL_PATH, device=DEVICE)
print(f'Loaded encoder on device: {active_device}')

results_df = benchmark_foldcomp_sample(
    encoder=encoder,
    foldcomp_db=foldcomp_db,
    sample_ids=sample_ids,
    out_dir=OUT_DIR,
    repeats=REPEATS,
    chunk_size=CHUNK_SIZE,
    queue_size=QUEUE_SIZE,
    batch_size=BATCH_SIZE,
    cache_size=CACHE_SIZE,
    verbose=False,
)

results_df

Loaded encoder on device: cuda


KeyboardInterrupt: 

In [ ]:
summary = estimate_total_runtime(
    total_entries=total_entries,
    throughputs=results_df['structures_per_second'].tolist(),
)

print('Throughput summary (structures/s):')
print(f"  mean: {summary['mean_throughput']:.2f}")
print(f"  std : {summary['std_throughput']:.2f}")
print(f"  95% CI: [{summary['throughput_ci95_low']:.2f}, {summary['throughput_ci95_high']:.2f}]")

print('\nEstimated full-database conversion time:')
print(f"  best case : {format_duration(summary['eta_best_seconds'])}")
print(f"  mean case : {format_duration(summary['eta_mean_seconds'])}")
print(f"  worst case: {format_duration(summary['eta_worst_seconds'])}")

summary

Throughput summary (structures/s):
  mean: 2.60
  std : 0.01
  95% CI: [2.59, 2.60]

Estimated full-database conversion time:
  best case : 2d 9h 50m 28s
  mean case : 2d 10h 0m 37s
  worst case: 2d 10h 10m 50s


{'mean_throughput': 2.5971334967518174,
 'std_throughput': 0.0067125574303925975,
 'throughput_ci95_low': 2.589537522946617,
 'throughput_ci95_high': 2.6047294705570176,
 'eta_mean_seconds': 208837.1663136844,
 'eta_worst_seconds': 209449.7550986756,
 'eta_best_seconds': 208228.1504205553}

In [12]:
# Test multiprocessing Foldcomp encoding on the same sampled IDs
MP_WORKERS = 8
MP_START_METHOD = "spawn"  # try "fork" on Linux if spawn is unstable in your setup
MP_OUT_DIR = f"{OUT_DIR}_mp"

mp_out_path = Path(MP_OUT_DIR)
mp_out_path.mkdir(parents=True, exist_ok=True)

mp_records = []
for run_idx in range(1, REPEATS + 1):
    fasta_path = mp_out_path / f"sample_run_{run_idx}.fasta"
    start = time.perf_counter()
    encoder.encode_foldcomp_fasta_mp(
        foldcomp_db=foldcomp_db,
        filename=str(fasta_path),
        ids=sample_ids,
        max_structures=None,
        chunk_size=20,
        queue_size=500,
        batch_size=BATCH_SIZE,
        cache_size=CACHE_SIZE,
        num_workers=MP_WORKERS,
        start_method=MP_START_METHOD,
        replace=True,
        alphabet=None,
        verbose=True,
    )
    elapsed = time.perf_counter() - start

    converted = len(sample_ids)
    throughput = converted / elapsed if elapsed > 0 else float("inf")
    fasta_size_mb = fasta_path.stat().st_size / (1024 ** 2)

    mp_records.append({
        "run": run_idx,
        "structures": converted,
        "seconds": elapsed,
        "structures_per_second": throughput,
        "fasta_size_mb": fasta_size_mb,
        "fasta_path": str(fasta_path),
    })

mp_results_df = pd.DataFrame.from_records(mp_records)
mp_results_df

if "results_df" in globals():
    baseline_tp = results_df["structures_per_second"].mean()
    mp_tp = mp_results_df["structures_per_second"].mean()
    speedup = mp_tp / baseline_tp if baseline_tp > 0 else float("inf")
    print(f"\nBaseline mean throughput: {baseline_tp:.2f} structures/s")
    print(f"MP mean throughput      : {mp_tp:.2f} structures/s")
    print(f"Speedup (MP / baseline) : {speedup:.2f}x")

Encoding Foldcomp DB to FASTA:   0%|          | 0/1000 [00:12<?, ?it/s]
Exception ignored in: <function BaseContext._insert_global.<locals>.on_disposal at 0x7f0c17687e50>
Traceback (most recent call last):
  File "/home/dmoi/miniforge3/envs/foldtree2/lib/python3.9/site-packages/numba/core/typing/context.py", line 518, in on_disposal
    def on_disposal(wr, pop=self._globals.pop):
KeyboardInterrupt: 
Exception ignored on calling ctypes callback function: <function ExecutionEngine._raw_object_cache_notify at 0x7f46fac5c670>
Traceback (most recent call last):
  File "/home/dmoi/miniforge3/envs/foldtree2/lib/python3.9/site-packages/llvmlite/binding/executionengine.py", line 178, in _raw_object_cache_notify
    def _raw_object_cache_notify(self, data):
KeyboardInterrupt: 
Process SpawnProcess-16:
Process SpawnProcess-9:
Process SpawnProcess-10:
Process SpawnProcess-12:
Process SpawnProcess-15:
Traceback (most recent call last):
  File "/home/dmoi/miniforge3/envs/foldtree2/lib/python3.9/mult

KeyboardInterrupt: 

egen.py", line 567, in _ensure_finalized
    self.finalize()
  File "/home/dmoi/projects/foldtree2/foldtree2/src/pdbgraphmk2.py", line 1116, in struct2pyg
    feat = self.create_features(pdbchain, foldcomp_data=foldcomp_data)
  File "/home/dmoi/miniforge3/envs/foldtree2/lib/python3.9/site-packages/numba/core/codegen.py", line 762, in finalize
    self._optimize_final_module()
  File "/home/dmoi/projects/foldtree2/foldtree2/src/pdbgraphmk2.py", line 1033, in create_features
    contact_points = self._compute_contact_matrix(arr['cb'], distance_cutoff=distance_cutoff)
  File "/home/dmoi/miniforge3/envs/foldtree2/lib/python3.9/site-packages/numba/core/codegen.py", line 682, in _optimize_final_module
    self._codegen._mpm_full.run(self._final_module)
  File "/home/dmoi/projects/foldtree2/foldtree2/src/pdbgraphmk2.py", line 780, in _compute_contact_matrix
    dmat = ret_distmat(xyz)
  File "/home/dmoi/miniforge3/envs/foldtree2/lib/python3.9/site-packages/llvmlite/binding/passmanagers.py", l

pe(func, args, kws)
  File "/home/dmoi/miniforge3/envs/foldtree2/lib/python3.9/site-packages/numba/core/typing/context.py", line 248, in _resolve_user_function_type
    return func.get_call_type(self, args, kws)
  File "/home/dmoi/miniforge3/envs/foldtree2/lib/python3.9/site-packages/numba/core/types/functions.py", line 308, in get_call_type
    sig = temp.apply(nolitargs, nolitkws)
  File "/home/dmoi/miniforge3/envs/foldtree2/lib/python3.9/site-packages/numba/core/typing/templates.py", line 350, in apply
    sig = generic(args, kws)
  File "/home/dmoi/miniforge3/envs/foldtree2/lib/python3.9/site-packages/numba/core/typing/templates.py", line 613, in generic
    disp, new_args = self._get_impl(args, kws)
  File "/home/dmoi/miniforge3/envs/foldtree2/lib/python3.9/site-packages/numba/core/typing/templates.py", line 712, in _get_impl
    impl, args = self._build_impl(cache_key, args, kws)
  File "/home/dmoi/miniforge3/envs/foldtree2/lib/python3.9/site-packages/numba/core/typing/templates.

## Optional: launch full conversion once estimate looks acceptable

You can run the full conversion in this notebook or from CLI.

Notebook path:

```python
FULL_OUTPUT = 'tmp/foldcomp_full_encoded.fasta'
encoder.encode_foldcomp_fasta(
    foldcomp_db=foldcomp_db,
    filename=FULL_OUTPUT,
    ids=None,
    max_structures=None,
    chunk_size=CHUNK_SIZE,
    queue_size=QUEUE_SIZE,
    batch_size=BATCH_SIZE,
    cache_size=CACHE_SIZE,
    replace=True,
    verbose=True,
)
```

CLI path:

```bash
python foldtree2/foldcomp2fasta.py \
  models/notebook/final_30char_contacts_aa_encoder_full_epoch_53.pt \
  /mnt/data1/foldcomp/afdb_swissprot_v4 \
  tmp/foldcomp_full_encoded.fasta \
  --chunk-size 1024 --queue-size 4 --batch-size 16 --cache-size 0
```

In [ ]:
# Profile conversion stage timings for PDB -> graph and Foldcomp -> graph
import io
import cProfile
import pstats
import time
import statistics
from pathlib import Path

from foldtree2.src import pdbgraphmk2

PROFILE_REPEATS = 3
ENABLE_CPROFILE = True
PDB_TEST_PATH = Path('/home/dmoi/projects/foldtree2/foldtree2/config/1eei.pdb')
FOLDCOMP_SAMPLE_ID = sample_ids[0] if 'sample_ids' in globals() and len(sample_ids) > 0 else None

converter_prof = pdbgraphmk2.PDB2PyG()

def summarize_timing(step_name: str, fn, repeats: int = PROFILE_REPEATS):
    times = []
    for _ in range(repeats):
        t0 = time.perf_counter()
        fn()
        times.append(time.perf_counter() - t0)
    return {
        'step': step_name,
        'repeats': repeats,
        'mean_s': statistics.mean(times),
        'stdev_s': statistics.stdev(times) if len(times) > 1 else 0.0,
        'min_s': min(times),
        'max_s': max(times),
    }

timing_rows = []

# ----- PDB path timings -----
if PDB_TEST_PATH.exists():
    pdb_path_str = str(PDB_TEST_PATH)
    timing_rows.append(summarize_timing('pdb.read_structure', lambda: converter_prof.read_structure(pdb_path_str)))
    timing_rows.append(summarize_timing('pdb.create_features', lambda: converter_prof.create_features(pdb_path_str)))
    timing_rows.append(summarize_timing('pdb.struct2pyg', lambda: converter_prof.struct2pyg(pdb_path_str)))
else:
    print(f'PDB test file not found: {PDB_TEST_PATH}')

# ----- Foldcomp path timings -----
fc_name = None
fc_payload = None
fc_data = None
if 'foldcomp_db' in globals() and FOLDCOMP_SAMPLE_ID is not None:
    import foldcomp

    timing_rows.append(
        summarize_timing(
            'foldcomp.open+fetch_one',
            lambda: next(iter(foldcomp.open(foldcomp_db, ids=[FOLDCOMP_SAMPLE_ID]))),
        )
    )

    with foldcomp.open(foldcomp_db, ids=[FOLDCOMP_SAMPLE_ID]) as db:
        fc_name, fc_payload = next(iter(db))

    timing_rows.append(summarize_timing('foldcomp.get_data', lambda: foldcomp.get_data(fc_payload)))
    fc_data = foldcomp.get_data(fc_payload)

    timing_rows.append(
        summarize_timing(
            'foldcomp.create_features(payload)',
            lambda: converter_prof.create_features(fc_payload, foldcomp_data=fc_data),
        )
    )
    timing_rows.append(
        summarize_timing(
            'foldcomp.struct2pyg(payload)',
            lambda: converter_prof.struct2pyg(fc_payload, identifier=fc_name, foldcomp_data=fc_data),
        )
    )
else:
    print('Skipping Foldcomp profiling: foldcomp_db/sample_ids are not available in this kernel.')

timing_df = pd.DataFrame(timing_rows).sort_values('mean_s', ascending=False).reset_index(drop=True)
timing_df

if ENABLE_CPROFILE and PDB_TEST_PATH.exists():
    print('\nTop cumulative-time functions for pdb.struct2pyg (single call):')
    prof = cProfile.Profile()
    prof.enable()
    converter_prof.struct2pyg(str(PDB_TEST_PATH))
    prof.disable()

    s = io.StringIO()
    pstats.Stats(prof, stream=s).sort_stats('cumtime').print_stats(25)
    print(s.getvalue())


Top cumulative-time functions for pdb.struct2pyg (single call):
         82073 function calls (80984 primitive calls) in 0.118 seconds

   Ordered by: cumulative time
   List reduced from 887 to 25 due to restriction <25>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        1    0.001    0.001    0.118    0.118 /home/dmoi/projects/foldtree2/foldtree2/src/pdbgraphmk2.py:1123(struct2pyg)
        1    0.000    0.000    0.105    0.105 /home/dmoi/projects/foldtree2/foldtree2/src/pdbgraphmk2.py:1017(create_features)
        1    0.001    0.001    0.040    0.040 /home/dmoi/projects/foldtree2/foldtree2/src/pdbgraphmk2.py:555(_extract_chain_arrays)
4169/3213    0.004    0.000    0.036    0.000 {built-in method numpy.core._multiarray_umath.implement_array_function}
      296    0.006    0.000    0.033    0.000 /home/dmoi/projects/foldtree2/foldtree2/src/pdbgraphmk2.py:195(_dihedral_rad)
        1    0.027    0.027    0.031    0.031 /home/dmoi/projects/foldtree2/foldt

In [ ]:
# Compact summary of the most expensive profiled steps
if 'timing_df' in globals() and len(timing_df) > 0:
    cols = ['step', 'mean_s', 'min_s', 'max_s']
    print('Top timing steps by mean_s:')
    print(timing_df[cols].sort_values('mean_s', ascending=False).to_string(index=False))
else:
    print('timing_df not found; run the profiling cell first.')

# Optional: show a concise cProfile top-10 if available
if 'prof' in globals():
    import io, pstats
    s2 = io.StringIO()
    pstats.Stats(prof, stream=s2).sort_stats('cumtime').print_stats(10)
    print('\nTop 10 cProfile cumulative-time entries:')
    print(s2.getvalue())

Top timing steps by mean_s:
                             step   mean_s    min_s    max_s
     foldcomp.struct2pyg(payload) 3.642453 3.625075 3.675516
foldcomp.create_features(payload) 3.552861 3.544036 3.561330
          foldcomp.open+fetch_one 0.272637 0.261725 0.278473
                   pdb.struct2pyg 0.090729 0.088425 0.092187
              pdb.create_features 0.075703 0.073689 0.077569
                foldcomp.get_data 0.013209 0.012607 0.014397
               pdb.read_structure 0.002561 0.002285 0.003004

Top 10 cProfile cumulative-time entries:
         82073 function calls (80984 primitive calls) in 0.118 seconds

   Ordered by: cumulative time
   List reduced from 887 to 10 due to restriction <10>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        1    0.001    0.001    0.118    0.118 /home/dmoi/projects/foldtree2/foldtree2/src/pdbgraphmk2.py:1123(struct2pyg)
        1    0.000    0.000    0.105    0.105 /home/dmoi/projects/foldtree2/foldtree2/src

In [ ]:
# Fine-grained stage timing inside pdbgraphmk2.create_features
from collections import defaultdict
import importlib
import time
from foldtree2.src import pdbgraphmk2

# Reset module state in case previous monkey-patching changed method descriptors in-kernel
pdbgraphmk2 = importlib.reload(pdbgraphmk2)

FG_REPEATS = 3

method_targets = [
    '_extract_chain_arrays',
    '_apply_foldcomp_backbone',
    '_angles_from_foldcomp',
    '_gemmi_angles',
    'add_aaproperties',
    '_compute_contact_matrix',
    '_compute_ss',
    '_fft_tracks',
    '_compute_bond_type_maps',
]

func_targets = [
    '_neighbor_stats',
    '_make_range_bins',
    '_make_burial_bins',
    '_make_bend_bins',
    '_make_torsion_bins',
    '_compute_wcn',
    '_dihedral_rad',
    '_compute_chi_angles',
]

def _run_fine_grained_profile(scenario_name, payload, foldcomp_data=None, repeats=FG_REPEATS):
    timings = defaultdict(lambda: [0.0, 0])  # name -> [total_seconds, calls]
    originals = {}

    def add_time(name, dt):
        timings[name][0] += dt
        timings[name][1] += 1

    def make_method_wrapper(name, fn):
        def wrapped(self, *args, **kwargs):
            t0 = time.perf_counter()
            try:
                return fn(self, *args, **kwargs)
            finally:
                add_time(f'method::{name}', time.perf_counter() - t0)
        return wrapped

    def make_func_wrapper(name, fn):
        def wrapped(*args, **kwargs):
            t0 = time.perf_counter()
            try:
                return fn(*args, **kwargs)
            finally:
                add_time(f'func::{name}', time.perf_counter() - t0)
        return wrapped

    # Patch class instance methods
    for name in method_targets:
        if not hasattr(pdbgraphmk2.PDB2PyG, name):
            continue
        orig_fn = getattr(pdbgraphmk2.PDB2PyG, name)
        originals[('method', name)] = orig_fn
        setattr(pdbgraphmk2.PDB2PyG, name, make_method_wrapper(name, orig_fn))

    # Patch module-level helper functions
    for name in func_targets:
        if hasattr(pdbgraphmk2, name):
            orig = getattr(pdbgraphmk2, name)
            originals[('func', name)] = orig
            setattr(pdbgraphmk2, name, make_func_wrapper(name, orig))

    converter_fg = pdbgraphmk2.PDB2PyG()

    total_wall = 0.0
    try:
        for _ in range(repeats):
            t0 = time.perf_counter()
            converter_fg.create_features(payload, foldcomp_data=foldcomp_data)
            total_wall += (time.perf_counter() - t0)
    finally:
        # Restore patched symbols
        for (kind, name), orig in originals.items():
            if kind == 'method':
                setattr(pdbgraphmk2.PDB2PyG, name, orig)
            else:
                setattr(pdbgraphmk2, name, orig)

    rows = []
    tracked_total = sum(v[0] for v in timings.values())
    for stage, (total_s, calls) in timings.items():
        rows.append({
            'scenario': scenario_name,
            'stage': stage,
            'total_s': total_s,
            'calls': calls,
            'mean_ms_per_call': (1000.0 * total_s / calls) if calls else 0.0,
            'pct_of_tracked': (100.0 * total_s / tracked_total) if tracked_total > 0 else 0.0,
        })

    stage_df = pd.DataFrame(rows).sort_values('total_s', ascending=False).reset_index(drop=True)
    summary = pd.DataFrame([{
        'scenario': scenario_name,
        'repeats': repeats,
        'wall_total_s': total_wall,
        'wall_mean_s': total_wall / repeats if repeats else float('nan'),
        'tracked_total_s': tracked_total,
    }])
    return stage_df, summary

all_stage_dfs = []
all_summaries = []

# PDB fine-grained profile
if 'PDB_TEST_PATH' in globals() and PDB_TEST_PATH.exists():
    stage_df_pdb, summary_pdb = _run_fine_grained_profile('pdb.create_features', str(PDB_TEST_PATH), foldcomp_data=None, repeats=FG_REPEATS)
    all_stage_dfs.append(stage_df_pdb)
    all_summaries.append(summary_pdb)
else:
    print('PDB_TEST_PATH is missing; skipping PDB fine-grained profiling.')

# Foldcomp fine-grained profile
if 'fc_payload' in globals() and fc_payload is not None and 'fc_data' in globals() and fc_data is not None:
    stage_df_fc, summary_fc = _run_fine_grained_profile('foldcomp.create_features(payload)', fc_payload, foldcomp_data=fc_data, repeats=FG_REPEATS)
    all_stage_dfs.append(stage_df_fc)
    all_summaries.append(summary_fc)
else:
    print('fc_payload/fc_data are missing; run profiling cell first to populate them.')

if all_summaries:
    fg_summary_df = pd.concat(all_summaries, ignore_index=True)
    print('Fine-grained profiling summary:')
    print(fg_summary_df.to_string(index=False))

if all_stage_dfs:
    fg_stage_df = pd.concat(all_stage_dfs, ignore_index=True)
    print('\nTop stages per scenario:')
    top_stage_df = fg_stage_df.groupby('scenario', group_keys=False).head(12)
    print(top_stage_df[['scenario', 'stage', 'total_s', 'calls', 'mean_ms_per_call', 'pct_of_tracked']].to_string(index=False))

fg_stage_df if all_stage_dfs else None

Fine-grained profiling summary:
                         scenario  repeats  wall_total_s  wall_mean_s  tracked_total_s
              pdb.create_features        3      0.229927     0.076642         0.314495
foldcomp.create_features(payload)        3     10.733081     3.577694        11.788743

Top stages per scenario:
                         scenario                            stage  total_s  calls  mean_ms_per_call  pct_of_tracked
              pdb.create_features  method::_compute_bond_type_maps 0.086450      3         28.816706       27.488544
              pdb.create_features    method::_extract_chain_arrays 0.074290      3         24.763289       23.621949
              pdb.create_features              func::_dihedral_rad 0.058959    888          0.066395       18.747108
              pdb.create_features        func::_compute_chi_angles 0.048145    309          0.155809       15.308640
              pdb.create_features         func::_make_torsion_bins 0.020968      3          6.98

,scenario,stage,total_s,calls,mean_ms_per_call,pct_of_tracked
0,pdb.create_features,method::_compute_bond_type_maps,0.086450,3,28.816706,27.488544
1,pdb.create_features,method::_extract_chain_arrays,0.074290,3,24.763289,23.621949
2,pdb.create_features,func::_dihedral_rad,0.058959,888,0.066395,18.747108
3,pdb.create_features,func::_compute_chi_angles,0.048145,309,0.155809,15.308640
4,pdb.create_features,func::_make_torsion_bins,0.020968,3,6.989257,6.667122
5,pdb.create_features,method::_fft_tracks,0.007108,3,2.369242,2.260044
6,pdb.create_features,method::add_aaproperties,0.005611,3,1.870454,1.784245
7,pdb.create_features,func::_neighbor_stats,0.003910,3,1.303239,1.243172
8,pdb.create_features,method::_gemmi_angles,0.003588,3,1.195932,1.140811
9,pdb.create_features,func::_compute_wcn,0.002303,3,0.767577,0.732200
